<a href="https://colab.research.google.com/github/Jayku88/22AIE301_Probabilistic_Reasoning/blob/main/22AIE301_Lab_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 22AIE301 - Probabilistic Reasoning
## Lab 08 - Junction Tree Algorithm

| | |
|---|---|
| **Name** | _________________________________ |
| **Roll No** | _____________ |
| **Date** | _____________ |
| **Lab Slot** | _____________ |

## Learning Objectives

By the end of this lab, you should be able to:

- Explain, with code, why sum-product belief propagation is undefined on graphs with cycles
- Construct a junction tree from an MRF: triangulate, extract maximal cliques, build a clique tree
- Implement clique-tree (sum-product) calibration and run it on a two-clique tree and a three-clique chain
- Verify the running intersection property (RIP) and cross-check overlapping clique beliefs
- Relate treewidth and triangulation quality to the size of the largest clique table
- Check every exact marginal you compute against brute-force enumeration

**Reference:** Koller & Friedman, *Probabilistic Graphical Models* (2009), Chapter 10.

**Academic integrity note:** all potential tables in this lab are generated from your roll
number via `np.random.seed(ROLL)`. Your numbers will differ from your neighbour's — the
*procedure* is what's being graded, not a specific number you can copy.

---
## Part 0 — Setup and Roll-Number Parameterization

Fill in your roll number below (as an integer — strip letters, e.g. `CH.SC.U4CSE23012` → `23012`).
Every potential table in this lab is derived from this seed, so re-running the notebook always
reproduces *your* numbers.

In [ ]:
import numpy as np
import networkx as nx
from itertools import product as iproduct

ROLL = ___   # TODO: your roll number as an integer
np.random.seed(ROLL)

print(f"Seeded with ROLL = {ROLL}")

---
## Part 1 — Why Plain BP Breaks on Cycles

Recall BP's **readiness rule**: a node may send to neighbour $j$ only once it has heard from
*every other* neighbour.

**Q1.1.** On the 4-cycle $A-B-C-D-A$ below, every node has exactly two neighbours. Trace through
the readiness rule by hand and explain, in 2-3 sentences, why no node is ever ready to send a
first message.

*(Your answer here.)*

___

**Q1.2.** "Loopy BP" ignores the readiness rule and forces every node to send an initial message
anyway, then keeps updating. Why are the resulting numbers, in general, not the correct
marginals? (One sentence is enough — think about what happens to a variable's influence as it
circulates around the cycle.)

*(Your answer here.)*

___

In [ ]:
# A small sanity-check function: given an undirected graph as a dict of adjacency lists,
# verify that no node has "all neighbours heard from" at the start (i.e. every node has
# degree >= 2 and the graph is a single cycle). Not graded — just a warm-up with networkx.

cycle_graph = nx.Graph()
cycle_graph.add_edges_from([("A", "B"), ("B", "C"), ("C", "D"), ("D", "A")])

for node in cycle_graph.nodes:
    degree = cycle_graph.degree[node]
    print(f"{node}: degree {degree}  ->  needs {degree} messages in hand before it can send")

is_tree = nx.is_tree(cycle_graph)
print(f"\nIs this graph a tree? {is_tree}  (BP as defined for trees is only valid when this is True)")

---
## Part 2 — A Generic `Factor` Class

Just as in Lab 07 (Variable Elimination) and Lab 08 (Belief Propagation), we represent every
potential / message / belief as a `Factor`: a set of variables, their cardinalities, and a
numpy array of values indexed by those variables.

Two operations do all the work in this lab:

- **`product`**: multiply two factors together (the "gather" half of a clique's job)
- **`marginalize`**: sum out one or more variables (the "sum-out" half)

Complete the two `TODO` blocks below.

In [ ]:
class Factor:
    def __init__(self, variables, cardinalities, values):
        """
        variables:      tuple/list of variable names, e.g. ('A', 'B')
        cardinalities:  dict mapping every variable name -> its cardinality
        values:         array-like, reshaped to match the variable order given
        """
        self.variables = tuple(variables)
        self.cardinalities = dict(cardinalities)
        shape = [self.cardinalities[v] for v in self.variables]
        self.values = np.array(values, dtype=float).reshape(shape)

    def get(self, assignment):
        """assignment: dict var -> value. Returns the single table entry."""
        idx = tuple(assignment[v] for v in self.variables)
        return self.values[idx]

    def product(self, other):
        """Return a new Factor over the UNION of self.variables and other.variables,
        whose value at any joint assignment is self.get(...) * other.get(...).
        """
        all_vars = list(self.variables) + [v for v in other.variables if v not in self.variables]
        card = {**self.cardinalities, **other.cardinalities}
        shape = [card[v] for v in all_vars]
        new_values = np.zeros(shape)

        for idx in np.ndindex(*shape):
            assignment = dict(zip(all_vars, idx))
            # TODO: multiply the two factors' entries at this assignment
            new_values[idx] = ___

        return Factor(all_vars, card, new_values)

    def marginalize(self, vars_to_sum_out):
        """Sum out one variable (str) or several (list/tuple of str).
        Returns a new Factor over the REMAINING variables.
        """
        if isinstance(vars_to_sum_out, str):
            vars_to_sum_out = [vars_to_sum_out]

        # TODO: compute the axis positions (in self.variables) of every variable
        # in vars_to_sum_out, and the tuple of variables that remain after removing them
        axes = ___
        remaining_vars = ___

        new_values = self.values.sum(axis=tuple(axes))
        return Factor(remaining_vars, self.cardinalities, new_values)

    def normalize(self):
        """Return a new Factor whose values sum to 1."""
        return Factor(self.variables, self.cardinalities, self.values / self.values.sum())

    def __repr__(self):
        return f"Factor{self.variables}\n{self.values}"


# --- Sanity checks (do not edit) ---------------------------------------------------
_f1 = Factor(("X", "Y"), {"X": 2, "Y": 2}, [[1, 2], [3, 4]])
_f2 = Factor(("Y", "Z"), {"Y": 2, "Z": 2}, [[1, 0], [0, 1]])

_prod = _f1.product(_f2)
assert set(_prod.variables) == {"X", "Y", "Z"}, "product() should union the variable sets"
assert np.isclose(_prod.get({"X": 0, "Y": 0, "Z": 0}), 1 * 1), "product() entry mismatch"
assert np.isclose(_prod.get({"X": 1, "Y": 1, "Z": 0}), 4 * 0), "product() entry mismatch"

_marg = _f1.marginalize("Y")
assert _marg.variables == ("X",), "marginalize() should drop the summed-out variable"
assert np.allclose(_marg.values, [3, 7]), "marginalize() should sum over the given axis"

print("Factor class: product() and marginalize() look correct.")

---
## Part 3 — Your 4-Cycle MRF

We reuse the running example from lecture: four binary variables $A, B, C, D$ arranged in a
cycle, with pairwise potentials $\phi_{AB}, \phi_{BC}, \phi_{CD}, \phi_{DA}$.

$$p(a, b, c, d) = \frac{1}{Z}\, \phi_{AB}(a,b)\, \phi_{BC}(b,c)\, \phi_{CD}(c,d)\, \phi_{DA}(d,a)$$

Run the cell below to generate **your** potential tables (random small positive integers,
seeded by your roll number).

In [ ]:
def random_potential(vars_pair, low=1, high=6):
    vals = np.random.randint(low, high, size=(2, 2))
    return Factor(vars_pair, {vars_pair[0]: 2, vars_pair[1]: 2}, vals)

phi_AB = random_potential(("A", "B"))
phi_BC = random_potential(("B", "C"))
phi_CD = random_potential(("C", "D"))
phi_DA = random_potential(("D", "A"))

for name, f in [("phi_AB", phi_AB), ("phi_BC", phi_BC), ("phi_CD", phi_CD), ("phi_DA", phi_DA)]:
    print(name, "=")
    print(f.values, "\n")

---
## Part 4 — Brute-Force Ground Truth

Before building the junction tree, compute the exact marginals the slow way — full joint
enumeration over all $2^4 = 16$ assignments. Every result in the rest of this lab must match
these numbers.

In [ ]:
def brute_force_joint(factors, variables):
    """Multiply every factor in `factors` together over the full variable set,
    returning the normalized joint Factor.
    """
    joint = factors[0]
    for f in factors[1:]:
        # TODO: accumulate the running product
        joint = ___
    return joint.normalize()

def brute_force_marginal(joint_factor, var):
    """Given the full normalized joint Factor, return p(var) by summing out everyone else."""
    other_vars = [v for v in joint_factor.variables if v != var]
    # TODO: sum out every variable except `var`
    return ___

joint_mrf1 = brute_force_joint([phi_AB, phi_BC, phi_CD, phi_DA], ["A", "B", "C", "D"])

bf_pA = brute_force_marginal(joint_mrf1, "A")
bf_pB = brute_force_marginal(joint_mrf1, "B")
bf_pC = brute_force_marginal(joint_mrf1, "C")
bf_pD = brute_force_marginal(joint_mrf1, "D")

for name, f in [("p(A)", bf_pA), ("p(B)", bf_pB), ("p(C)", bf_pC), ("p(D)", bf_pD)]:
    print(f"{name} = {f.values.round(4)}")

assert np.isclose(joint_mrf1.values.sum(), 1.0), "The full joint must sum to 1"
print("\nBrute-force ground truth computed — keep these numbers for later comparison.")

---
## Part 5 — Step 2: Triangulate

The 4-cycle $A-B-C-D-A$ has no chord, so it is **not chordal**. Add the chord $A$–$C$ (exactly
the fill-in VE would create eliminating $B$ or $D$ first) and verify chordality with
`networkx`.

In [ ]:
raw_graph = nx.Graph()
raw_graph.add_edges_from([("A", "B"), ("B", "C"), ("C", "D"), ("D", "A")])
print("Raw 4-cycle chordal?", nx.is_chordal(raw_graph))

triangulated = raw_graph.copy()
# TODO: add the chord A-C to make the graph chordal
___

print("After adding chord A-C, chordal?", nx.is_chordal(triangulated))
assert nx.is_chordal(triangulated), "Graph should be chordal after adding one chord to a 4-cycle"


---
## Part 6 — Step 3: Extract Maximal Cliques

`networkx.find_cliques` enumerates all maximal cliques of a graph. Use it to confirm the two
maximal cliques of the triangulated graph, and read off the induced width $w$.

In [ ]:
maximal_cliques = [frozenset(c) for c in nx.find_cliques(triangulated)]
print("Maximal cliques found:", maximal_cliques)

# TODO: the induced width is (size of the LARGEST maximal clique) - 1
induced_width = ___
print("Induced width w =", induced_width)

assert len(maximal_cliques) == 2, "Triangulating a 4-cycle with one chord should give exactly 2 maximal cliques"
assert induced_width == 2, "Both cliques should have size 3, so w = 2"
assert frozenset({"A", "B", "C"}) in maximal_cliques
assert frozenset({"A", "C", "D"}) in maximal_cliques

---
## Part 7 — Step 4: Build the Clique Tree and Verify RIP

With only two cliques, the clique tree is a single edge. The separator is the intersection of
the two cliques.

**Running Intersection Property (RIP):** for every variable $x$, the set of cliques containing
$x$ must form a *connected* subtree. On a two-node tree this is automatic, but you will check it
properly in Part 9 (three cliques) — write the general checker now so you can reuse it.

In [ ]:
C1 = frozenset({"A", "B", "C"})
C2 = frozenset({"A", "C", "D"})
sep_12 = C1 & C2
print("Separator C1 ∩ C2 =", sep_12)

def check_rip(clique_list, tree_edges):
    """clique_list: list of frozensets (the cliques).
    tree_edges: list of (i, j) index-pairs into clique_list that form the tree.
    For every variable, the cliques containing it must be connected in the tree.
    Returns True if RIP holds, else False.
    """
    tree_graph = nx.Graph()
    tree_graph.add_nodes_from(range(len(clique_list)))
    tree_graph.add_edges_from(tree_edges)

    all_vars = set().union(*clique_list)
    for var in all_vars:
        # TODO: find the indices of every clique containing `var`
        containing = ___
        if len(containing) <= 1:
            continue
        # TODO: check that the induced subgraph on `containing` is connected
        induced = tree_graph.subgraph(containing)
        if not nx.is_connected(induced):
            return False
    return True

cliques_ex1 = [C1, C2]
tree_edges_ex1 = [(0, 1)]
assert check_rip(cliques_ex1, tree_edges_ex1), "RIP should hold for this two-clique tree"
print("RIP verified for the two-clique tree.")

---
## Part 8 — Worked Example 1: Calibrate the Two-Clique Tree

$$C_1 = \{A,B,C\},\ \psi_1 = \phi_{AB}\,\phi_{BC} \qquad C_2 = \{A,C,D\},\ \psi_2 = \phi_{CD}\,\phi_{DA}$$

Assign every original factor to the (unique) clique whose scope contains it, then run
collect + distribute over the single edge.

In [ ]:
# --- Assign factors and form initial potentials -----------------------------------
# TODO: psi1 = phi_AB * phi_BC   (both factors have scope subset {A,B,C})
psi1 = ___
# TODO: psi2 = phi_CD * phi_DA   (both factors have scope subset {A,C,D})
psi2 = ___

print("psi1 (A,B,C):\n", psi1.values)
print("psi2 (A,C,D):\n", psi2.values)

In [ ]:
# --- Collect: C1 is a leaf, sends first --------------------------------------------
# TODO: m1_to_2(a,c) = sum_b psi1(a,b,c)
m1_to_2 = ___

# C2 has now heard from its only neighbour -> form its belief and read off p(D)
b2 = psi2.product(m1_to_2)
# TODO: p(D) by summing b2 over A and C, then normalizing
p_D = ___

print("m1_to_2 (A,C):\n", m1_to_2.values)
print("\np(D) =", p_D.values.round(4))

In [ ]:
# --- Distribute: C2 sends back to C1 -----------------------------------------------
# TODO: m2_to_1(a,c) = sum_d psi2(a,c,d)
m2_to_1 = ___

b1 = psi1.product(m2_to_1)
# TODO: p(B) by summing b1 over A and C, then normalizing
p_B = ___

print("m2_to_1 (A,C):\n", m2_to_1.values)
print("\np(B) =", p_B.values.round(4))

In [ ]:
# --- Cross-check: A and C appear in BOTH cliques, RIP guarantees they must agree ---
p_A_from_b1 = b1.marginalize(["B", "C"]).normalize()
p_A_from_b2 = b2.marginalize(["C", "D"]).normalize()
p_C_from_b1 = b1.marginalize(["A", "B"]).normalize()
p_C_from_b2 = b2.marginalize(["A", "D"]).normalize()

assert np.allclose(p_A_from_b1.values, p_A_from_b2.values, atol=1e-8), "p(A) must agree across cliques"
assert np.allclose(p_C_from_b1.values, p_C_from_b2.values, atol=1e-8), "p(C) must agree across cliques"
print("p(A) and p(C) agree across C1 and C2, as RIP requires.")

# --- Compare against Part 4's brute-force ground truth -----------------------------
assert np.allclose(p_A_from_b1.values, bf_pA.values, atol=1e-8), "p(A) should match brute force"
assert np.allclose(p_B.values, bf_pB.values, atol=1e-8), "p(B) should match brute force"
assert np.allclose(p_C_from_b1.values, bf_pC.values, atol=1e-8), "p(C) should match brute force"
assert np.allclose(p_D.values, bf_pD.values, atol=1e-8), "p(D) should match brute force"
print("All four junction-tree marginals match brute-force enumeration exactly.")

---
## Part 9 — Worked Example 2: A Three-Clique Chain

Attach a new binary variable $E$ to $D$ by a single edge (a pendant). The triangulation of the
$A,B,C,D$ part is unaffected — only $E$ adds a new size-2 clique $\{D,E\}$.

$$C_1=\{A,B,C\} \ - \ C_2=\{A,C,D\} \ - \ C_3=\{D,E\}$$

Now $C_2$ has **two** tree-neighbours, so it must wait for both before it is ready.

In [ ]:
phi_DE = random_potential(("D", "E"))
print("phi_DE =\n", phi_DE.values)

C3 = frozenset({"D", "E"})
sep_23 = C2 & C3
print("Separator C2 ∩ C3 =", sep_23)

cliques_ex2 = [C1, C2, C3]
tree_edges_ex2 = [(0, 1), (1, 2)]
assert check_rip(cliques_ex2, tree_edges_ex2), "RIP should hold for this three-clique chain"
print("RIP verified for the three-clique chain.")

In [ ]:
# --- Brute-force ground truth for the 5-variable model (2^5 = 32 assignments) -----
joint_mrf2 = brute_force_joint([phi_AB, phi_BC, phi_CD, phi_DA, phi_DE],
                                ["A", "B", "C", "D", "E"])
bf2 = {v: brute_force_marginal(joint_mrf2, v) for v in ["A", "B", "C", "D", "E"]}
for v in ["A", "B", "C", "D", "E"]:
    print(f"p({v}) = {bf2[v].values.round(4)}")

In [ ]:
# --- Collect phase: both leaves (C1 and C3) send first -----------------------------
psi3 = phi_DE  # only one factor has scope subset {D, E}

# m1_to_2 is IDENTICAL to Part 8 -- the {A,B,C} side of the graph hasn't changed.
# (No need to recompute; reuse the m1_to_2 variable from Part 8.)

# TODO: m3_to_2(d) = sum_e psi3(d,e)
m3_to_2 = ___

print("m1_to_2 (reused):\n", m1_to_2.values)
print("m3_to_2:\n", m3_to_2.values)

In [ ]:
# --- C2 is now ready (heard from both neighbours): belief and p(D) -----------------
b2_ex2 = psi2.product(m1_to_2).product(m3_to_2)
p_D_ex2 = b2_ex2.marginalize(["A", "C"]).normalize()
print("p(D) =", p_D_ex2.values.round(4))
assert np.allclose(p_D_ex2.values, bf2["D"].values, atol=1e-8), "p(D) should match brute force"


In [ ]:
# --- Distribute phase: C2 sends OUTWARD in both directions ------------------------
# Toward C1: combine psi2 with ONLY the message from the OTHER side (C3)
# TODO: m2_to_1(a,c) = sum_d psi2(a,c,d) * m3_to_2(d)
m2_to_1_ex2 = ___

# Toward C3: combine psi2 with ONLY the message from the OTHER side (C1)
# TODO: m2_to_3(d) = sum_{a,c} psi2(a,c,d) * m1_to_2(a,c)
m2_to_3 = ___

print("m2_to_1 (A,C):\n", m2_to_1_ex2.values)
print("m2_to_3 (D):\n", m2_to_3.values)

In [ ]:
# --- Final beliefs and remaining marginals -----------------------------------------
b1_ex2 = psi1.product(m2_to_1_ex2)
b3_ex2 = psi3.product(m2_to_3)

p_A_ex2 = b1_ex2.marginalize(["B", "C"]).normalize()
p_B_ex2 = b1_ex2.marginalize(["A", "C"]).normalize()
p_C_ex2 = b1_ex2.marginalize(["A", "B"]).normalize()
p_E_ex2 = b3_ex2.marginalize(["D"]).normalize()

for name, computed, truth in [
    ("A", p_A_ex2, bf2["A"]), ("B", p_B_ex2, bf2["B"]), ("C", p_C_ex2, bf2["C"]),
    ("E", p_E_ex2, bf2["E"]),
]:
    print(f"p({name}) junction tree = {computed.values.round(4)}   brute force = {truth.values.round(4)}")
    assert np.allclose(computed.values, truth.values, atol=1e-8), f"p({name}) mismatch"

# Sanity check: Z must match across all three clique beliefs
Z1, Z2, Z3 = b1_ex2.values.sum(), b2_ex2.values.sum(), b3_ex2.values.sum()
print(f"\nZ from C1={Z1:.4f}, C2={Z2:.4f}, C3={Z3:.4f}")
assert np.isclose(Z1, Z2) and np.isclose(Z2, Z3), "All clique beliefs must share the same normalizer Z"
print("\nAll five marginals match brute-force enumeration. Junction tree is exact.")

---
## Part 10 — In-Class Exercise: Choosing a Triangulation

Back to the plain 4-cycle $A-B-C-D-A$ (ignore $E$ for this part). Two students propose different
fixes:

- **Student X:** add chord $B$–$D$ only
- **Student Y:** add *both* chords $A$–$C$ **and** $B$–$D$

For each proposal: is the result chordal? What are the maximal cliques and the induced width
$w$? Which would you recommend, and why?

In [ ]:
# --- Student X: chord B-D only -------------------------------------------------
graph_X = nx.Graph()
graph_X.add_edges_from([("A", "B"), ("B", "C"), ("C", "D"), ("D", "A")])
# TODO: add Student X's chord
___

chordal_X = nx.is_chordal(graph_X)
cliques_X = [frozenset(c) for c in nx.find_cliques(graph_X)]
width_X = max(len(c) for c in cliques_X) - 1

print("Student X — chordal?", chordal_X)
print("Student X — maximal cliques:", cliques_X)
print("Student X — induced width w =", width_X)

In [ ]:
# --- Student Y: both chords A-C and B-D -----------------------------------------
graph_Y = nx.Graph()
graph_Y.add_edges_from([("A", "B"), ("B", "C"), ("C", "D"), ("D", "A")])
# TODO: add BOTH of Student Y's chords
___

chordal_Y = nx.is_chordal(graph_Y)
cliques_Y = [frozenset(c) for c in nx.find_cliques(graph_Y)]
width_Y = max(len(c) for c in cliques_Y) - 1

print("Student Y — chordal?", chordal_Y)
print("Student Y — maximal cliques:", cliques_Y)
print("Student Y — induced width w =", width_Y)

assert chordal_X and chordal_Y, "Both proposals should be chordal"
assert width_X == 2, "Student X's triangulation should have width 2"
assert width_Y == 3, "Student Y's triangulation should have width 3 (one 4-way clique)"


**Q10.1.** Compute the total number of table entries each proposal requires
(sum of $2^{|\text{clique}|}$ over all maximal cliques). Which proposal is cheaper, and by how
much?

*(Your answer here — you may add a code cell to compute this.)*

___

**Q10.2.** Both proposals give a chordal, correct junction tree. Why would you still prefer one
over the other? Relate your answer to Lecture 8's elimination-ordering comparison.

*(Your answer here.)*

___

In [ ]:
# TODO (Q10.1): compute and print the total table-entry cost for each proposal
cost_X = ___
cost_Y = ___
print(f"Student X total entries: {cost_X}")
print(f"Student Y total entries: {cost_Y}")

---
## Part 11 — Reflection Questions

**Q11.1.** In Part 9, the message $m_{2\to3}$ you computed should equal the *unnormalized*
$p(D)$ you found in Part 8's Worked Example 1 (up to the normalizer $Z=130$-style constant from
lecture — here it will be your own roll-seeded numbers). Verify this with an assertion, and
explain in one sentence *why* this is not a coincidence.

*(Your answer here.)*

___

**Q11.2.** Suppose you needed only $p(E)$ and nothing else, under no evidence. Would plain
Variable Elimination or the full junction tree construction be the more sensible choice here?
Justify your answer using the "Good fit / Poor fit" criteria from lecture.

*(Your answer here.)*

___

**Q11.3.** If the 4-cycle were instead a dense graph where every triangulation produces a clique
of size close to $n$, what would you do instead of running the junction tree algorithm?

*(Your answer here.)*

___

In [ ]:
# TODO (Q11.1): verify m2_to_3 matches Part 8's unnormalized p(D) belief
unnormalized_pD_ex1 = b2.marginalize(["A", "C"])  # from Part 8, before normalizing
assert np.allclose(m2_to_3.values, unnormalized_pD_ex1.values, atol=1e-8), \
    "m2_to_3 should equal the unnormalized p(D) computed with only the A,B,C-side factors"
print("Confirmed: m2_to_3 exactly reproduces Part 8's unnormalized p(D).")

---
## Part 12 — Summary

Run the cell below as a final self-check. If every assertion in this notebook has passed, all
of your junction-tree marginals for both worked examples are provably exact, cross-checked
against brute-force enumeration.

In [ ]:
print("="*60)
print(f"Lab 09 self-check — ROLL = {ROLL}")
print("="*60)
print("Example 1 (4-cycle, two cliques):")
print(f"  p(A) = {p_A_from_b1.values.round(4)}")
print(f"  p(B) = {p_B.values.round(4)}")
print(f"  p(C) = {p_C_from_b1.values.round(4)}")
print(f"  p(D) = {p_D.values.round(4)}")
print("\nExample 2 (4-cycle + pendant E, three cliques):")
print(f"  p(A) = {p_A_ex2.values.round(4)}")
print(f"  p(B) = {p_B_ex2.values.round(4)}")
print(f"  p(C) = {p_C_ex2.values.round(4)}")
print(f"  p(D) = {p_D_ex2.values.round(4)}")
print(f"  p(E) = {p_E_ex2.values.round(4)}")
print("\nAll assertions above passed -> junction tree results are exact.")

---
## Submission Checklist
-  Your **Roll No** is set correctly in the Setup cell (`ROLL`).
-  Save the notebook: **File → Save** (or Ctrl+S).

**Rename the file as:** `Lab08_<YourRollNo>.ipynb` and convert to PDF before submitting.